# Machine Fault Recognition

## Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

import librosa
import librosa.display 
import IPython.display as ipd

import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

from itertools import cycle

sns.set_theme(style="white", palette=None)
color_pal = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])



# !pip install wandb

import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

## Preprocessing Helper Functions 

In [ ]:
def create_resampler(orig_sr: int, new_sr: int):
    resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=new_sr)
    return resampler

def resample(audio: torch.Tensor, resampler) -> torch.Tensor:
    audio = resampler(audio)
    return audio


def peak_normalize(audio: torch.Tensor) -> torch.Tensor: 
    max_val = torch.max(torch.abs(audio))

    if max_val == 0:
        return audio
    
    return audio / max_val


def rms_normalize(audio: torch.Tensor) -> torch.Tensor: 
    rms = torch.sqrt(torch.mean(audio * audio))
    return audio / (rms + 1e-8)


def cut(audio: torch.Tensor, sr: int, lower_cut=0.5, upper_cut=9.5) -> torch.Tensor:
    start = int(lower_cut * sr)
    end = int(upper_cut * sr)
    audio = audio[start:end]
    return audio

## Feature Extraction Helper Functions

In [ ]:

def extract_mfcc(audio: torch.Tensor, sr, n_mfcc=13):
    mfcc_transform = torchaudio.transforms.MFCC(sample_rate=sr, n_mfcc=n_mfcc,
        melkwargs={
            "n_fft": 1024,
            "hop_length": 512,
            "n_mels": 40
        }
    )

    return mfcc_transform(audio)


def mfcc_stats(mfcc):

    mean = mfcc.mean(dim=1)
    std = mfcc.std(dim=1)

    return torch.cat([mean, std])


def mfcc_stats_with_deltas(mfcc):

    delta = F.compute_deltas(mfcc)
    delta2 = F.compute_deltas(delta)

    return torch.cat([
        mfcc.mean(dim=1), mfcc.std(dim=1),
        delta.mean(dim=1), delta.std(dim=1),
        delta2.mean(dim=1), delta2.std(dim=1)
    ])


def extract_spectrogram(audio: torch.Tensor, n_fft=1024, hop_length=512):
    # audio shape must be [1, T]
    if audio.dim() == 1:
        audio = audio.unsqueeze(0)

    transform = torchaudio.transforms.Spectrogram(
        n_fft=n_fft,
        hop_length=hop_length,
        power=2.0
    )

    S = transform(audio)  # [1, freq_bins, time]

    S_db = torchaudio.transforms.AmplitudeToDB()(S)

    return S_db.squeeze(0)  # [freq_bins, time]    


def extract_mel_spectrogram(audio: torch.Tensor, sr: int,
                                   n_fft=1024, hop_length=512, n_mels=40):

    if audio.dim() == 1:
        audio = audio.unsqueeze(0)

    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0
    )

    mel = mel_transform(audio)

    mel_db = torchaudio.transforms.AmplitudeToDB()(mel)

    return mel_db.squeeze(0)  # [n_mels, time]

## Data Loading

Machine 1 + Normal     → 0

Machine 1 + Abnormal   → 1

Machine 2 + Normal     → 2

Machine 2 + Abnormal   → 3

Machine 3 + Normal     → 4

Machine 3 + Abnormal   → 5

In [ ]:
DATA_PATH = "/kaggle/input/datasets/mmagdy908/machine-audio-for-pattern-recognition"

label_map = {
    ("Machine 1", "Normal"): 0,
    ("Machine 1", "Abnormal"): 1,
    ("Machine 2", "Normal"): 2,
    ("Machine 2", "Abnormal"): 3,
    ("Machine 3", "Normal"): 4,
    ("Machine 3", "Abnormal"): 5,
}

file_paths = []
labels = []

for machine in ["Machine 1", "Machine 2", "Machine 3"]:
    for condition in ["Normal", "Abnormal"]:
        
        folder = os.path.join(
            DATA_PATH,
            machine,
            "machine_data",
            condition
        )
        
        files = glob.glob(os.path.join(folder, "*.wav"))
        
        for f in files:
            file_paths.append(f)
            labels.append(label_map[(machine, condition)])

### Confirm Data Loading

In [ ]:
print(f"Total files: {len(file_paths)}")

pd.Series(labels).value_counts()

## Data Splitting

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(file_paths, labels, test_size=0.3, random_state=42, stratify=labels)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(len(X_train), len(X_val), len(X_test))

| Train       | Validation | Test |
|------------|--------|--------|
| 39,365       | 8,435   | 8,436   |


## GPU Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Explore Data

### Load a sample from each class

In [ ]:
data = sorted(zip(file_paths, labels), key=lambda x: x[1])
selected = {}

for path, label in zip(file_paths, labels):
    if label not in selected:
        selected[label] = path

y_machine1_normal, sr_machine1_normal = librosa.load(selected[0])
y_machine1_abnormal, sr_machine1_abnormal = librosa.load(selected[1])

y_machine2_normal, sr_machine2_normal = librosa.load(selected[2])
y_machine2_abnormal, sr_machine2_abnormal = librosa.load(selected[3])

y_machine3_normal, sr_machine3_normal = librosa.load(selected[4])
y_machine3_abnormal, sr_machine3_abnormal = librosa.load(selected[5])

### Extract MFCC

In [ ]:
mfcc_normal_1 = librosa.feature.mfcc(y=y_machine1_normal, sr=sr_machine1_normal, n_mfcc=13)
mfcc_abnormal_1 = librosa.feature.mfcc(y=y_machine1_abnormal, sr=sr_machine1_abnormal, n_mfcc=13)

mfcc_normal_2 = librosa.feature.mfcc(y=y_machine2_normal, sr=sr_machine2_normal, n_mfcc=13)
mfcc_abnormal_2 = librosa.feature.mfcc(y=y_machine2_abnormal, sr=sr_machine2_abnormal, n_mfcc=13)

mfcc_normal_3 = librosa.feature.mfcc(y=y_machine3_normal, sr=sr_machine3_normal, n_mfcc=13)
mfcc_abnormal_3 = librosa.feature.mfcc(y=y_machine3_abnormal, sr=sr_machine3_abnormal, n_mfcc=13)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 10))

# Machine 1
img1 = librosa.display.specshow(
    mfcc_normal_1, x_axis='time', sr=sr_machine1_normal, ax=axes[0, 0]
)
axes[0, 0].set_title('Machine 1 - Normal')
fig.colorbar(img1, ax=axes[0, 0])

img2 = librosa.display.specshow(
    mfcc_abnormal_1, x_axis='time', sr=sr_machine1_abnormal, ax=axes[0, 1]
)
axes[0, 1].set_title('Machine 1 - Abnormal')
fig.colorbar(img2, ax=axes[0, 1])

# Machine 2
img3 = librosa.display.specshow(
    mfcc_normal_2, x_axis='time', sr=sr_machine2_normal, ax=axes[1, 0]
)
axes[1, 0].set_title('Machine 2 - Normal')
fig.colorbar(img3, ax=axes[1, 0])

img4 = librosa.display.specshow(
    mfcc_abnormal_2, x_axis='time', sr=sr_machine2_abnormal, ax=axes[1, 1]
)
axes[1, 1].set_title('Machine 2 - Abnormal')
fig.colorbar(img4, ax=axes[1, 1])

# Machine 3
img5 = librosa.display.specshow(
    mfcc_normal_3, x_axis='time', sr=sr_machine3_normal, ax=axes[2, 0]
)
axes[2, 0].set_title('Machine 3 - Normal')
fig.colorbar(img5, ax=axes[2, 0])

img6 = librosa.display.specshow(
    mfcc_abnormal_3, x_axis='time', sr=sr_machine3_abnormal, ax=axes[2, 1]
)
axes[2, 1].set_title('Machine 3 - Abnormal')
fig.colorbar(img6, ax=axes[2, 1])

plt.tight_layout()
plt.show()

### Extract Spectograms and Mel-Spectograms

#### Spectograms

In [ ]:
def extract_spectrogram(audio, n_fft=1024, hop_length=512):
    S = librosa.stft(audio)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    return S_db

def extract_mel_spectrogram(audio, sr, n_fft=1024, hop_length=512, n_mels=40):
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

fig, axes = plt.subplots(3, 2, figsize=(18, 10))

# Machine 1
img1 = librosa.display.specshow(
    extract_spectrogram(y_machine1_normal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine1_normal,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Machine 1 - Normal (Spectrogram)')
fig.colorbar(img1, ax=axes[0, 0])

img2 = librosa.display.specshow(
    extract_spectrogram(y_machine1_abnormal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine1_abnormal,
    ax=axes[0, 1]
)
axes[0, 1].set_title('Machine 1 - Abnormal (Spectrogram)')
fig.colorbar(img2, ax=axes[0, 1])

# Machine 2
img3 = librosa.display.specshow(
    extract_spectrogram(y_machine2_normal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine2_normal,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Machine 2 - Normal (Spectrogram)')
fig.colorbar(img3, ax=axes[1, 0])

img4 = librosa.display.specshow(
    extract_spectrogram(y_machine2_abnormal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine2_abnormal,
    ax=axes[1, 1]
)
axes[1, 1].set_title('Machine 2 - Abnormal (Spectrogram)')
fig.colorbar(img4, ax=axes[1, 1])

# Machine 3
img5 = librosa.display.specshow(
    extract_spectrogram(y_machine3_normal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine3_normal,
    ax=axes[2, 0]
)
axes[2, 0].set_title('Machine 3 - Normal (Spectrogram)')
fig.colorbar(img5, ax=axes[2, 0])

img6 = librosa.display.specshow(
    extract_spectrogram(y_machine3_abnormal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine3_abnormal,
    ax=axes[2, 1]
)
axes[2, 1].set_title('Machine 3 - Abnormal (Spectrogram)')
fig.colorbar(img6, ax=axes[2, 1])

plt.tight_layout()
plt.show()

#### Mel-Spectograms

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 10))

# Machine 1
img1 = librosa.display.specshow(
    extract_mel_spectrogram(y_machine1_normal, sr_machine1_normal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine1_normal,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Machine 1 - Normal (Mel-Spectrogram)')
fig.colorbar(img1, ax=axes[0, 0])

img2 = librosa.display.specshow(
    extract_mel_spectrogram(y_machine1_abnormal, sr_machine1_abnormal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine1_abnormal,
    ax=axes[0, 1]
)
axes[0, 1].set_title('Machine 1 - Abnormal (Mel-Spectrogram)')
fig.colorbar(img2, ax=axes[0, 1])

# Machine 2
img3 = librosa.display.specshow(
    extract_mel_spectrogram(y_machine2_normal, sr_machine2_normal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine2_normal,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Machine 2 - Normal (Mel-Spectrogram)')
fig.colorbar(img3, ax=axes[1, 0])

img4 = librosa.display.specshow(
    extract_mel_spectrogram(y_machine2_abnormal, sr_machine2_abnormal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine2_abnormal,
    ax=axes[1, 1]
)
axes[1, 1].set_title('Machine 2 - Abnormal (Mel-Spectrogram)')
fig.colorbar(img4, ax=axes[1, 1])

# Machine 3
img5 = librosa.display.specshow(
    extract_mel_spectrogram(y_machine3_normal, sr_machine3_normal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine3_normal,
    ax=axes[2, 0]
)
axes[2, 0].set_title('Machine 3 - Normal (Mel-Spectrogram)')
fig.colorbar(img5, ax=axes[2, 0])

img6 = librosa.display.specshow(
    extract_mel_spectrogram(y_machine3_abnormal, sr_machine3_abnormal),
    x_axis='time',
    y_axis="hz",
    sr=sr_machine3_abnormal,
    ax=axes[2, 1]
)
axes[2, 1].set_title('Machine 3 - Abnormal (Mel-Spectrogram)')
fig.colorbar(img6, ax=axes[2, 1])

plt.tight_layout()
plt.show()

## Dataset Class

In [ ]:
class MachineAudioDataset(Dataset):
    def __init__(self, file_paths, labels, preprocessor, extractor):
        self.file_paths = file_paths
        self.labels = labels
        self.preprocessor = preprocessor
        self.extractor = extractor       

    def __len__(self):
        return len(self.file_paths)
    
    def load_audio(self, path):
        audio, sr = torchaudio.load(path)
        return audio, sr

        
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]

        audio, sr = self.load_audio(path)

        if audio.shape[0] > 1:
            audio = audio.mean(dim=0)
        else:
            audio = audio.squeeze(0)       

        audio = self.preprocessor.resample(audio.unsqueeze(0), sr)
        audio = self.preprocessor.cut(audio)
        audio = self.preprocessor.normalize(audio)   

        feat = self.extractor.extract(audio)
        feat = feat.unsqueeze(0)
        
        return feat, torch.tensor(label)

### Create Dataset

In [ ]:
# train_dataset = MachineAudioDataset(X_train, y_train)
# val_dataset   = MachineAudioDataset(X_val, y_val)
# test_dataset  = MachineAudioDataset(X_test, y_test)

### Create Data Loader

In [ ]:
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
# test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Feature Extraction

In [ ]:
import torch
import torchaudio
import torchaudio.functional as F


class FeatureExtractor:
    def __init__(self, sr=16000, feature_type="mel"):
        self.sr = sr
        self.feature_type = feature_type

        self.mfcc_transform = torchaudio.transforms.MFCC(
            sample_rate=sr,
            n_mfcc=13,
            melkwargs={
                "n_fft": 1024,
                "hop_length": 512,
                "n_mels": 40
            }
        )

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=1024,
            hop_length=512,
            n_mels=64,
            power=2.0
        )

        self.spec_transform = torchaudio.transforms.Spectrogram(
            n_fft=1024,
            hop_length=512,
            power=2.0
        )

        self.db_transform = torchaudio.transforms.AmplitudeToDB()


    def mfcc(self, audio):
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)

        mfcc = self.mfcc_transform(audio)
        return mfcc.squeeze(0)

    def mfcc_stats(self, audio):
        mfcc = self.mfcc(audio)

        mean = mfcc.mean(dim=1)
        std = mfcc.std(dim=1)

        return torch.cat([mean, std])

    def mfcc_stats_with_deltas(self, audio):
        mfcc = self.mfcc(audio)

        delta = F.compute_deltas(mfcc)
        delta2 = F.compute_deltas(delta)

        return torch.cat([
            mfcc.mean(dim=1), mfcc.std(dim=1),
            delta.mean(dim=1), delta.std(dim=1),
            delta2.mean(dim=1), delta2.std(dim=1)
        ])


    def spectrogram(self, audio):
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)

        spec = self.spec_transform(audio)
        spec_db = self.db_transform(spec)

        return spec_db.squeeze(0)


    def mel_spectrogram(self, audio):
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)

        mel = self.mel_transform(audio)
        mel_db = self.db_transform(mel)

        return mel_db.squeeze(0)


    def extract(self, audio):
        if self.feature_type == "mfcc":
            return self.mfcc_stats_with_deltas(audio)

        elif self.feature_type == "mfcc_simple":
            return self.mfcc_stats(audio)

        elif self.feature_type == "mel":
            return self.mel_spectrogram(audio)

        elif self.feature_type == "spec":
            return self.spectrogram(audio)

        else:
            raise ValueError("Unknown feature type")

## Preprocessoring

In [ ]:
class AudioPreprocessor:
    def __init__(self, orig_sr=22050, target_sr=16000, mode="peak"):
        self.target_sr = target_sr
        self.orig_sr = orig_sr
        self.mode = mode
        self.resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=target_sr)

    def resample(self, audio, sr):
        if sr != self.target_sr:
            audio = self.resampler(audio)
        return audio

    def cut(self, audio):
        start = int(0.5 * self.target_sr)
        end = int(9.5 * self.target_sr)
        return audio[:, start:end]

    def normalize(self, audio):
        if self.mode == "peak":
            max_val = torch.max(torch.abs(audio))
            return audio / (max_val + 1e-8)

        elif self.mode == "rms":
            rms = torch.sqrt(torch.mean(audio ** 2))
            return audio / (rms + 1e-8)

        else:
            return audio

## ML Models

### Evaluation Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
import numpy as np


def evaluate_model(model, X, y, num_classes=6):


    y_pred = model.predict(X)


    cm = confusion_matrix(y, y_pred)

    plt.figure(figsize=(6,6))
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title("Confusion Matrix")
    
    wandb.log({"confusion_matrix": wandb.Image(plt)})

    results = {
        "Accuracy": accuracy_score(y, y_pred),


        "Precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y, y_pred, average="macro"),
        "F1_macro": f1_score(y, y_pred, average="macro"),

        "Precision_weighted": precision_score(y, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y, y_pred, average="weighted"),
        "F1_weighted": f1_score(y, y_pred, average="weighted"),


        "MCC": matthews_corrcoef(y, y_pred)
    }

    return results

### Building Features

In [ ]:
X_train_feats = []
y_train_feats = []

preprocessor = AudioPreprocessor()
extractor = FeatureExtractor(feature_type="mfcc")

for path, label in tqdm(zip(X_train, y_train), total=len(X_train)):
    audio, sr = torchaudio.load(path)
    audio = audio.mean(dim=0)

    # preprocessing
    audio = preprocessor.resample(audio.unsqueeze(0), sr)
    audio = preprocessor.cut(audio)
    audio = preprocessor.normalize(audio)

    # feature extraction
    feat = extractor.extract(audio)

    X_train_feats.append(feat.numpy())
    y_train_feats.append(label)

X_train_feats = np.array(X_train_feats)
y_train_feats = np.array(y_train_feats)

In [ ]:
np.save("X_train_feats.npy", X_train_feats)
np.save("y_train_feats.npy", y_train_feats)

In [ ]:
X_val_feats = []
y_val_feats = []

preprocessor = AudioPreprocessor()
extractor = FeatureExtractor(feature_type="mfcc")

for path, label in tqdm(zip(X_val, y_val), total=len(X_val)):
    audio, sr = torchaudio.load(path)
    audio = audio.mean(dim=0)

    # preprocessing
    audio = preprocessor.resample(audio.unsqueeze(0), sr)
    audio = preprocessor.cut(audio)
    audio = preprocessor.normalize(audio)

    # feature extraction
    feat = extractor.extract(audio)

    X_val_feats.append(feat.numpy())
    y_val_feats.append(label)

X_val_feats = np.array(X_val_feats)
y_val_feats = np.array(y_val_feats)

In [ ]:
np.save("X_val_feats.npy", X_val_feats)
np.save("y_val_feats.npy", y_val_feats)

In [ ]:
X_test_feats = []
y_test_feats = []

preprocessor = AudioPreprocessor()
extractor = FeatureExtractor(feature_type="mfcc")

for path, label in tqdm(zip(X_test, y_test), total=len(X_test)):
    audio, sr = torchaudio.load(path)
    audio = audio.mean(dim=0)

    # preprocessing
    audio = preprocessor.resample(audio.unsqueeze(0), sr)
    audio = preprocessor.cut(audio)
    audio = preprocessor.normalize(audio)

    # feature extraction
    feat = extractor.extract(audio)

    X_test_feats.append(feat.numpy())
    y_test_feats.append(label)

X_test_feats = np.array(X_test_feats)
y_test_feats = np.array(y_test_feats)

In [ ]:
np.save("X_test_feats.npy", X_test_feats)
np.save("y_test_feats.npy", y_test_feats)

### LogisticRegression

In [ ]:
wandb.init(
    project="machine-fault-recognition",
    name="mfcc-lr-baseline",
    config={
        "feature_type": "mfcc_stats_with_deltas",
        "model": "Logistic Regression",
        "n_estimators": 200,
        "sr": 16000,
        "n_fft": 1024,
        "hop_length": 512,
        "n_mels": 40
    }
)

In [ ]:
from sklearn.linear_model import LogisticRegression

LR_model = LogisticRegression(max_iter=2000)
LR_model.fit(X_train_feats, y_train_feats)

results = evaluate_model(LR_model, X_val_feats, y_val_feats)
print(results)
wandb.log(results)

### RandomForestClassifier

In [ ]:
wandb.init(
    project="machine-fault-recognition",
    name="mfcc-rf-baseline",
    config={
        "feature_type": "mfcc_stats_with_deltas",
        "model": "RandomForest",
        "n_estimators": 200,
        "sr": 16000,
        "n_fft": 1024,
        "hop_length": 512,
        "n_mels": 40
    }
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

RF_model = RandomForestClassifier(n_estimators=200)
RF_model.fit(X_train_feats, y_train_feats)

results = evaluate_model(RF_model, X_val_feats, y_val_feats)

print(results)
wandb.log(results)

## CNN - Baseline - Experiment 1 

### Load Data

In [ ]:
preprocessor = AudioPreprocessor()
extractor = FeatureExtractor(sr=16000, feature_type="mel")

train_dataset = MachineAudioDataset(X_train, y_train, preprocessor, extractor)
val_dataset   = MachineAudioDataset(X_val, y_val, preprocessor, extractor)
test_dataset  = MachineAudioDataset(X_test, y_test, preprocessor, extractor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
for batch in train_loader:
    x, y = batch
    print("X shape:", x.shape)
    print("Y shape:", y.shape)
    break

### Model class

In [ ]:
import torch 
import torch.nn as nn

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        self.features = nn.Sequential(
            # (32, 1, 64, 282)
            #Block1
            nn.Conv2d(1, 16, kernel_size=3, padding=1), 
            nn.BatchNorm2d(16), 
            nn.ReLU(), 
            nn.MaxPool2d(2), 

            #Block2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), 
            nn.ReLU(), 
            nn.MaxPool2d(2), 

            #Block3
            nn.Conv2d(32, 64, kernel_size=3, padding=1), 
            nn.BatchNorm2d(64), 
            nn.ReLU(), 
            nn.MaxPool2d(2)

            #(32, 64, 8, 35)
        )

        self.classifier = nn.Sequential(
            #(32, 64, 8, 35)
            nn.AdaptiveAvgPool2d((1, 1)), #could be removed and uses the linear layer
            #(32, 64, 1, 1)
            nn.Flatten(),
            #(32, 64)
            nn.Linear(64, 128), 
            nn.ReLU(), 
            nn.Dropout(0.3), 
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x 


### Model Eval. Func.

In [ ]:
def evaluate(model, loader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for x, y in loop:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            loop.set_postfix(loss=loss.item())

    acc = correct / total
    return total_loss / len(loader), acc


def evaluate_model(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    y_pred = np.array(all_preds)
    y_true = np.array(all_labels)

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")

    wandb.log({"confusion_matrix": wandb.Image(plt)})
    plt.close()

    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro"),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "mcc": matthews_corrcoef(y_true, y_pred)
    }

    return results

### Training Function

In [ ]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(loader, desc="Training", leave=False)

    for x, y in loop:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

        # update tqdm bar
        loop.set_postfix(loss=loss.item())

    acc = correct / total
    return total_loss / len(loader), acc

### Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaselineCNN(num_classes=6).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

wandb.init(
    project="machine-fault-recognition",
    name="cnn-mel-baseline",
    config={
        "model": "BaselineCNN",
        "feature_type": "mel",
        "optimizer": "Adam",
        "lr": 1e-3,
        "batch_size": 32,
        "epochs": num_epochs
    }
)


train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    print("-" * 40)

In [ ]:
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.legend()
plt.title("Loss Curve")

wandb.log({"loss_curve": wandb.Image(plt)})
plt.close()

In [ ]:
final_results = evaluate_model(model, test_loader)

print("Final Results:")
for k, v in final_results.items():
    print(f"{k}: {v:.4f}")

wandb.log(final_results)

In [ ]:
torch.save(model.state_dict(), "baseline_cnn.pth")